# Day 02 · 고급 검색 전략 실습 (로컬)
방대한 사내 문서(정책 6건 + 제품 매뉴얼 50건, 약 5만 자)를 직접 **청킹·메타데이터 부착**해 벡터DB에 넣고,
슬라이드에서 배운 검색 기법을 **하나씩 코드로** 확인합니다 —
**메타데이터 필터 · BM25 · 하이브리드(RRF) · Reranker · HyDE · Multi-Query · MMR · 재배치 · 검색 평가(Recall·MRR) · GraphRAG**.
검색·재정렬은 전부 CPU 로컬이고, LLM은 **NVIDIA build**(Qwen)로 생성·질의 변환·관계 추출에만 씁니다.

## 0 · 설치 & 설정 (NVIDIA 토큰)
LLM은 **NVIDIA build** 를 씁니다. 개인 **NVIDIA API 토큰**(`nvapi-...`)이 필요해요 — 없으면 `실습_시작하기.ipynb` 에서 발급.
아래 설정 셀 실행 시 토큰을 입력받습니다(환경변수 `NVIDIA_API_KEY` 있으면 자동).

In [ ]:
!pip install -q sentence-transformers qdrant-client langchain-text-splitters openai rank_bm25 networkx matplotlib

In [ ]:
import os, getpass
from openai import OpenAI

# 생성 엔드포인트: NVIDIA build (기본). 강사 DGX로 바꾸려면 아래 줄 주석 해제.
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
# LLM_BASE_URL = "http://124.51.229.210:30001/v1"
# NVIDIA API 토큰: 환경변수(NVIDIA_API_KEY) 있으면 자동, 없으면 직접 입력(화면 비표시)
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")

_c = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in _c.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("엔드포인트:", LLM_BASE_URL, "| 모델:", LLM_MODEL)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

## 1 · 방대한 문서 확보 (파일에서 로딩)
예시 몇 줄이 아니라, 여러 섹션으로 된 실제 문서를 **텍스트 파일(`corpus.txt`)에서 읽어** 다룹니다.
정책 6건과 제품 매뉴얼 50건(약 5만 자)이 들어 있고, 각 문서는 메타데이터 헤더 한 줄과 본문으로 되어 있습니다.

확인 — 문서 56건이 파일에서 로드되고 각 문서가 메타데이터를 갖는가?

In [ ]:
import os, urllib.request

# corpus.txt: 정책 6건 + 제품 매뉴얼 50건(약 5만 자). 각 문서 = 메타데이터 헤더 한 줄 + 본문.
DATA_URL = "https://raw.githubusercontent.com/TunaLee/nvidia-dli/main/day02/data/corpus.txt"
cands = ["corpus.txt", "day02/data/corpus.txt", "data/corpus.txt"]
local = next((p for p in cands if os.path.exists(p)), None)
if local is None:                                  # Colab 등: 저장소에서 파일을 내려받는다
    local = "corpus.txt"
    urllib.request.urlretrieve(DATA_URL, local)
    print("corpus.txt 다운로드 완료")

raw = open(local, encoding="utf-8").read()
documents = []
for block in raw.split("===DOC==="):               # 문서 단위로 분리
    block = block.strip()
    if not block:
        continue
    head, _, body = block.partition("\n")          # 첫 줄 = 메타데이터, 나머지 = 본문
    meta = {}
    for kv in head.split(" | "):
        k, v = kv.split(": ", 1)
        meta[k.strip()] = v.strip()
    meta["year"] = int(meta["year"])
    documents.append({**meta, "body": body.strip()})

total = sum(len(d["body"]) for d in documents)
print(f"문서 {len(documents)}건 로드 · 총 {total:,}자 (평균 {total//len(documents):,}자/건) | 카테고리:",
      sorted(set(d["category"] for d in documents)))

### 청킹 + 메타데이터 부착
긴 본문을 그대로 넣으면 한 청크가 너무 커져 검색이 뭉툭해집니다. 문서를 청크로 쪼개고,
**모든 청크에 원본 문서의 메타데이터를 물려줍니다** — 이 메타데이터가 뒤의 필터 기준이 됩니다.

확인 — 문서 56건이 수백 개의 청크로 쪼개지고, 각 청크가 메타데이터를 갖는가?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = []
for d in documents:
    for j, p in enumerate(splitter.split_text(d["body"])):
        meta = {k: d[k] for k in ("doc_id", "title", "category", "department", "year")}
        if "product_id" in d:
            meta["product_id"] = d["product_id"]
        chunks.append({"text": p, "chunk_idx": j, **meta})

texts = [c["text"] for c in chunks]        # 이후 셀들이 쓰는 '청크 본문' 목록
print(f"총 청크 {len(chunks)}개 (문서 {len(documents)}건에서)")
print("예시 청크:", {k: chunks[300][k] for k in ("text", "category", "product_id") if k in chunks[300]})

### 임베딩 + 벡터DB (메타데이터를 payload로 저장)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

embedder = SentenceTransformer(EMBED_MODEL)     # 최초 1회 다운로드로 시간이 걸릴 수 있어요
def embed(xs): return embedder.encode(xs, normalize_embeddings=True)

chunk_vecs = embed(texts)                       # 청크 전체를 배치 임베딩 (MMR 등에서 재사용)
client = QdrantClient(":memory:")
dim = chunk_vecs.shape[1]
if client.collection_exists("docs"):
    client.delete_collection("docs")
client.create_collection("docs", vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
client.upsert("docs", points=[PointStruct(id=i, vector=chunk_vecs[i].tolist(), payload=chunks[i])
                              for i in range(len(chunks))])

def vector_search_idx(query, n=10):
    qv = embed([query])[0]
    return [p.id for p in client.query_points("docs", query=qv.tolist(), limit=n).points]

def show(idxs):                                 # 인덱스 → 내용 미리보기 (결과 확인용)
    return [texts[i][:22] for i in idxs]

print("저장 완료:", client.count("docs").count, "청크")
print("벡터 검색:", show(vector_search_idx("연차는 며칠 전에 신청하나요?", 3)))

## 2 · 메타데이터 필터링
메타데이터로 검색 범위를 좁힙니다. 같은 질문이라도 `category=제품`, `2024년 이후`처럼 조건을 걸면
원하는 문서 집합 안에서만 찾습니다.

확인 — 필터 없이/있을 때 결과 카테고리가 어떻게 달라지는가?

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

def search(query, k=5, category=None, department=None, min_year=None):
    conds = []
    if category:   conds.append(FieldCondition(key="category",   match=MatchValue(value=category)))
    if department: conds.append(FieldCondition(key="department", match=MatchValue(value=department)))
    if min_year:   conds.append(FieldCondition(key="year",       range=Range(gte=min_year)))
    flt = Filter(must=conds) if conds else None
    pts = client.query_points("docs", query=embed([query])[0].tolist(), limit=k, query_filter=flt).points
    return [(p.payload["category"], p.payload["year"], p.payload["text"][:26]) for p in pts]

print("[필터 없음] 펌웨어는 어떻게 업데이트하나요?")
for r in search("펌웨어는 어떻게 업데이트하나요?", 3): print("  ", r)
print("\n[category=인사 · 2024년 이후]")
for r in search("펌웨어는 어떻게 업데이트하나요?", 3, category="인사", min_year=2024): print("  ", r)

## Lab 1 · BM25 (키워드 검색)
정확 토큰(ERR-4021, X-137)에 강한 색인 검색. 한국어는 조사가 붙으므로 정규식으로 토큰을 살립니다.

확인 — `ERR-4021`이 BM25 상위로 잡히는가?

In [ ]:
import re
from rank_bm25 import BM25Okapi

# 한국어는 조사가 붙어 단순 split()이면 'ERR-4021은'·'X-137의' 같은 토큰이 되어 쿼리와 매칭되지 않습니다.
# 영문/숫자/하이픈 덩어리와 한글 덩어리를 분리해 '정확 토큰'을 살립니다.
def tokenize(t):
    return re.findall(r"[A-Za-z0-9\-]+|[가-힣]+", t)

bm25 = BM25Okapi([tokenize(t) for t in texts])

def bm25_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    return sorted(range(len(scores)), key=lambda i: -scores[i])[:k]

print("BM25 'ERR-4021':", show(bm25_search("ERR-4021", 3)))
print("BM25 'X-137 펌웨어':", show(bm25_search("X-137 펌웨어", 3)))

## Lab 2 · 하이브리드 + RRF
벡터(뜻)와 BM25(정확 토큰) 결과를 **순위(rank)** 로 합칩니다. k=60 (슬라이드 손계산과 같은 공식: 1위 → 1/61).

확인 — 벡터만 / BM25만 / 하이브리드의 상위가 어떻게 다른가?

In [ ]:
def rrf(rank_lists, k=60):
    score = {}
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            score[idx] = score.get(idx, 0) + 1/(k + r + 1)
    return sorted(score, key=lambda i: -score[i])

def hybrid(query, n=10):
    return rrf([vector_search_idx(query, n), bm25_search(query, n)])[:n]

for tag, fn in [("벡터", vector_search_idx), ("BM25", bm25_search), ("하이브리드", hybrid)]:
    print(f"{tag:5s} →", show(fn("X-137 펌웨어 버전", 3)))

## Lab 3 · Reranker (서류전형 → 면접)
cross-encoder가 (질문, 문서)를 **함께** 읽어 관련성으로 다시 줄세웁니다. 넓게 후보 N개 → 정밀하게 K개.

확인 — rerank 전/후 Top-3 순서가 어떻게 바뀌는가?

In [ ]:
from sentence_transformers import CrossEncoder
# 다국어 reranker (한국어 지원). 영어 전용 ms-marco-MiniLM 은 한국어에서 오작동합니다.
# 최초 1회 모델 다운로드로 몇십 초 걸릴 수 있어요 — 멈춘 게 아닙니다!
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def rerank(query, cand_idx, top_k=3):
    pairs = [(query, texts[i]) for i in cand_idx]
    scores = reranker.predict(pairs)
    order = sorted(range(len(cand_idx)), key=lambda j: -scores[j])
    return [cand_idx[j] for j in order[:top_k]]

def rerank_topk(query, k=3):                    # 검색 평가에서 쓰기 좋은 형태
    return rerank(query, hybrid(query, 12), top_k=k)

q = "비용은 누가 최종 승인하나요?"
cand = hybrid(q, n=12)               # 1차: 넓게 (서류전형)
best = rerank(q, cand, top_k=3)      # 2차: 정밀 (면접)
print("rerank 전:", show(cand[:3]))
print("rerank 후:", show(best))

### '유사도 1등'이 진짜 1등이 아닐 때
아래 질문은 "어디에서 접수하는가"를 묻습니다. 임베딩 유사도는 단어가 더 겹치는 *기한* 문장을 1위로 올리지만(오답),
크로스인코더 리랭커는 (질문, 문서)를 함께 읽어 의도를 파악하고 *접수 장소* 문장을 1위로 올립니다.

In [ ]:
demo_q = "비용 정산은 어디에서 접수하나요?"
demo_cands = [
    "비용 정산 신청은 지출일로부터 30일 이내에 완료해야 한다.",           # 단어는 겹치지만 '기한'
    "경비 청구는 사내포털의 정산 메뉴에서 접수하고 재무팀이 최종 승인한다.",  # 실제 '접수 장소'
    "재택근무는 주 2회까지 가능하다.",
]
sim = embed(demo_cands) @ embed([demo_q])[0]
rsc = reranker.predict([(demo_q, c) for c in demo_cands])
print("질문:", demo_q)
print("\n[임베딩 유사도 순위]  ← 1위가 '기한' 문장(오답)")
for i in np.argsort(-sim): print(f"  {sim[i]:.3f}  {demo_cands[i][:34]}")
print("\n[리랭커 순위]  ← 1위가 '접수 장소' 문장(정답)")
for i in np.argsort(-rsc): print(f"  {rsc[i]:6.2f}  {demo_cands[i][:34]}")

## Lab 4 · HyDE (가상 답변으로 검색)
짧고 모호한 질문에 '그럴듯한 가상 답'을 만들어 **그걸로** 검색합니다. 짧은 질문을 풍부한 문서에 가깝게 바꾸는 변환기입니다.

확인 — 같은 모호 질문에서 '질문 그대로' vs 'HyDE'의 결과 차이

In [ ]:
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)   # NVIDIA build (Qwen)

def ask(prompt, temperature=0.3):
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=temperature,
        messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    return out.split("</think>")[-1].strip()   # 모델이 <think>를 출력하는 설정이어도 안전

def hyde_search(query, n=5):
    hypo = ask(f"다음 질문에 짧게 그럴듯한 답을 써줘:\n{query}")
    return vector_search_idx(hypo, n)

q = "그거 며칠 전에 말해야 해요?"          # 짧고 모호한 질문
print("질문 그대로:", show(vector_search_idx(q, 3)))
print("HyDE 검색  :", show(hyde_search(q, 3)))

## Lab 5 · Multi-Query (여러 표현으로 검색)
질문을 여러 표현으로 바꿔 각각 검색 → RRF로 합칩니다. **원 질문도 목록에 포함**하는 게 관행입니다.

확인 — 변형 질의들이 서로 다른 청크를 데려오는가?

In [ ]:
def multi_query(query, n=6, verbose=True):
    raw = ask(f"질문을 뜻은 같되 표현이 다른 3개로 바꿔, 한 줄씩만 적어줘:\n{query}", temperature=0.7)
    # LLM이 붙이는 번호("1. ")·서두 문장("다음은 ~:")을 걷어냅니다
    lines = [re.sub(r'^\s*[\d\-\.\)\*]+\s*', '', l).strip() for l in raw.splitlines()]
    variants = [query] + [l for l in lines if l and not l.endswith(':')][:3]
    if verbose:
        print("검색에 쓸 표현들:", variants)
    return rrf([vector_search_idx(v, n) for v in variants])[:n]

print("결과:", show(multi_query("재택 언제까지 알려줘야 하나요?")))

## Lab 6 · MMR (다양성 — 중복 대신 폭을 준다)
**MMR = Maximal Marginal Relevance**(최대 한계 관련성). 후보를 고를 때 *질문과의 관련성*만 보지 않고,
**이미 뽑은 것과 얼마나 겹치는지**를 함께 봅니다 — 그래서 '같은 얘기 5개' 대신 '관련되면서 서로 다른 5개'를 담습니다.

점수식: `λ · (질문과의 관련도) − (1−λ) · (이미 뽑힌 것과의 최대 유사도)`.
λ가 1이면 관련도만(중복 허용), 0에 가까우면 다양성을 크게 봅니다.

확인 — 유사도 Top-3는 거의 같은 문장을 담고, MMR Top-3는 서로 다른 정보를 담는가?

In [ ]:
# 일부러 '거의 중복'과 '다른 정보'를 섞은 후보로 효과를 봅니다.
demo_q2 = "휴가는 어떻게 신청하나요?"
demo_docs2 = [
    "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다.",   # 신청(핵심)
    "휴가 신청은 사내포털 휴가 메뉴에서 3영업일 전에 하고 팀장이 승인한다.",       # 신청(거의 중복)
    "연차는 근속 연수에 따라 최대 25일까지 늘어난다.",                              # 다른 정보(일수)
    "반차와 반반차도 같은 절차로 신청할 수 있다.",                                  # 다른 정보(단위)
    "승인 없이 사용하면 결근으로 처리될 수 있다.",                                  # 다른 정보(주의)
]
dv = embed(demo_docs2); qv = embed([demo_q2])[0]
rel = dv @ qv

def mmr_select(doc_vecs, query_vec, k=3, lam=0.6):
    r = doc_vecs @ query_vec
    chosen, rest = [], list(range(len(doc_vecs)))
    while rest and len(chosen) < k:
        if not chosen:
            j = max(rest, key=lambda j: r[j])
        else:
            j = max(rest, key=lambda j: lam*r[j] - (1-lam)*max(doc_vecs[j] @ doc_vecs[s] for s in chosen))
        chosen.append(j); rest.remove(j)
    return chosen

print("[유사도 Top-3]  ← 거의 같은 '신청' 문장이 자리 차지")
for i in np.argsort(-rel)[:3]: print(f"  {rel[i]:.3f}  {demo_docs2[i]}")
print("\n[MMR Top-3]  ← 신청 + 일수 + 단위/주의로 폭이 넓어짐")
for i in mmr_select(dv, qv, 3): print(f"  {rel[i]:.3f}  {demo_docs2[i]}")

이제 실제 코퍼스에서 후보에 MMR을 적용하는 재사용 함수입니다. 미리 구한 `chunk_vecs`를 씁니다.

In [ ]:
def mmr(query, cand_idx, k=5, lam=0.7):
    qv = embed([query])[0]
    cv = np.array([chunk_vecs[i] for i in cand_idx])
    r = cv @ qv
    chosen, rest = [], list(range(len(cand_idx)))
    while rest and len(chosen) < k:
        if not chosen:
            j = max(rest, key=lambda j: r[j])
        else:
            j = max(rest, key=lambda j: lam*r[j] - (1-lam)*max(cv[j] @ cv[s] for s in chosen))
        chosen.append(j); rest.remove(j)
    return [cand_idx[j] for j in chosen]

q = "펌웨어는 어떻게 업데이트하나요?"          # 제품 50개가 같은 안내 문장을 공유 → 후보에 중복이 많음
cand = hybrid(q, 15)
top5 = cand[:5]
mmr5 = mmr(q, cand, 5, lam=0.5)
print("하이브리드 상위5의 서로 다른 내용 수:", len(set(texts[i] for i in top5)), "/ 5")
print("MMR 상위5의 서로 다른 내용 수      :", len(set(texts[i] for i in mmr5)), "/ 5  ← 중복이 줄어 폭이 넓어짐")
print("MMR 결과:", show(mmr5))

## Lab 7 · 재배치 (Lost in the Middle)
LLM은 긴 컨텍스트에서 **처음과 끝**을 잘 보고 가운데는 흐릿하게 봅니다('lost in the middle').
그래서 최종 K개를 넘기기 전에, **가장 강한 근거를 양 끝**에 두도록 순서를 바꿉니다.

확인 — 점수 1·2위가 목록의 처음과 끝으로 가는가?

In [ ]:
def reorder_ends(idx_scores):
    # (idx, score) 목록 → 점수 높은 순으로 정렬한 뒤 큰 것을 양 끝에 번갈아 배치
    ordered = [i for i, _ in sorted(idx_scores, key=lambda x: -x[1])]
    head, tail = [], []
    for rank, i in enumerate(ordered):
        (head if rank % 2 == 0 else tail).append(i)
    return head + tail[::-1]

q = "비용은 누가 최종 승인하나요?"
top = rerank(q, hybrid(q, 12), top_k=5)
sc = reranker.predict([(q, texts[i]) for i in top])
print("재배치 전(점수순):", [f"{s:.1f}" for s in sorted(sc, reverse=True)])
reordered = reorder_ends(list(zip(top, sc)))
print("재배치 후 순서    :", show(reordered))
print("→ 1·2위가 처음과 끝, 약한 근거는 가운데")

## Lab 8 · 검색 평가 — Recall@k · MRR
'느낌'이 아니라 숫자로 검색을 채점합니다. 정답 키워드가 상위 k개 청크 안에 들었으면 성공(Recall),
몇 번째에서 처음 맞았는지로 순위 품질(MRR)을 봅니다. 기법별로 비교합니다.

확인 — 비슷한 제품 50개에서 정확 토큰을 찾을 때 어떤 기법이 이기는가?

In [ ]:
EVALSET = [
    ("X-137의 펌웨어 최신 버전은?",        "3.37"),
    ("X-142의 기술 문의 담당자는 누구인가요?", "담당자43"),
    ("X-119의 정격 전압은 몇 V인가요?",     "4.9"),
    ("인증 토큰이 만료되면 뜨는 코드는?",     "ERR-4021"),
    ("비용 정산은 어디에서 접수하나요?",     "정산 메뉴"),
]
def eval_retrieval(fn, k=3):
    recall, mrr = 0, 0.0
    for q, kw in EVALSET:
        idxs = fn(q, k)
        hit = [r for r, i in enumerate(idxs) if kw in texts[i]]
        if hit:
            recall += 1; mrr += 1/(hit[0] + 1)
    n = len(EVALSET); return round(recall/n, 3), round(mrr/n, 3)

print(f"{'기법':<12}{'Recall@3':>10}{'MRR':>8}")
print("-"*30)
for name, fn in [("벡터", vector_search_idx), ("BM25", bm25_search), ("하이브리드", hybrid), ("리랭커", rerank_topk)]:
    rec, mrr = eval_retrieval(fn); print(f"{name:<12}{rec:>10}{mrr:>8}")

## Lab 9 · 최종 파이프라인 & A/B 비교
오늘 배운 걸 하나로 잇습니다: **하이브리드(N) → Reranker(정밀) → MMR(다양성) → 재배치 → 생성**.
모호 질문은 앞단에 **HyDE·Multi-Query**를 더합니다. 기준선(벡터 only)과 답을 비교합니다.

LLM 호출이 많아 몇 분 걸릴 수 있어요.

In [ ]:
SYSTEM = ("아래 [자료]의 내용만 근거로 답하라. 없으면 '문서에 없음'이라고만 답하라. "
          "답 끝에 (근거)를 표기하라.")

def generate(query, idxs):
    ctx = "\n".join(f"[{i+1}] {texts[d]}" for i, d in enumerate(idxs))
    out = llm.chat.completions.create(model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"[자료]\n{ctx}\n\n[질문] {query}"}])
    return out.choices[0].message.content.split("</think>")[-1].strip()

def advanced_rag(query, vague=False, diversify=True):
    cand = hybrid(query, n=12)
    if vague:   # 모호 질문: HyDE·Multi-Query로 후보 보강 (파이프라인 '맨 앞' 블록)
        cand = rrf([cand, hyde_search(query, 12), multi_query(query, 12, verbose=False)])[:12]
    top = rerank(query, cand, top_k=6)                 # 정밀 재정렬
    top = mmr(query, top, k=3) if diversify else top[:3]  # 다양성
    sc = reranker.predict([(query, texts[i]) for i in top])
    top = reorder_ends(list(zip(top, sc)))             # 재배치(강한 근거를 양 끝)
    return generate(query, top)

def baseline_rag(query):                                # Day 1 벡터-only 기준선
    return generate(query, vector_search_idx(query, 3))

In [ ]:
QUESTIONS = {
    "정확 토큰형": ["에러코드 ERR-4021 해결법은?", "X-137 펌웨어 최신 버전은?", "외부 USB 써도 되나요?"],
    "의미형":     ["로그인이 자꾸 풀리는데 왜죠?", "비용은 누가 최종 승인하나요?", "집에서 일하려면 어떻게 해야 하나요?"],
    "모호형":     ["그거 며칠 전에 말해야 해요?", "파일은 어디로 공유해요?"],
}
for cat, qs in QUESTIONS.items():
    for q in qs:
        print(f"\n◆ [{cat}] {q}")
        print("  기준선:", baseline_rag(q))
        print("  고급  :", advanced_rag(q, vague=(cat == "모호형")))

## Lab 10 · 정답 품질 평가 — LLM 심판
검색이 잘 됐는지(Recall·MRR)와, 최종 '답'이 실제로 맞았는지는 다른 문제입니다.
파이프라인이 낸 답을 기준 정답과 비교해 **LLM이 채점**합니다(맞으면 1, 아니면 0). 평균이 정답률입니다 — 조합을 바꿔 이 점수를 올리는 게 미니 프로젝트의 목표입니다.

확인 — 벡터-only 기준선과 리랭커의 정답률이 다른가?

In [ ]:
def judge(question, reference, answer):
    p = (f"질문: {question}\n기준 정답: {reference}\nRAG 답변: {answer}\n"
         "RAG 답변이 기준 정답과 사실상 일치하면 1, 아니면 0. 숫자만 출력.")
    return "1" in ask(p, temperature=0)

JUDGE_SET = [
    ("X-137의 펌웨어 최신 버전은?", "3.37"),
    ("인증 토큰이 만료되면 뜨는 코드는?", "ERR-4021"),
    ("비용은 누가 최종 승인하나요?", "재무팀"),
]
def answer_accuracy(retriever, k=3):
    correct = 0
    for q, ref in JUDGE_SET:
        a = generate(q, retriever(q, k))
        ok = judge(q, ref, a); correct += ok
        print(f"  [{'O' if ok else 'X'}] {q} -> {a[:36]}")
    return round(correct / len(JUDGE_SET), 3)

print("벡터-only 정답률 :", answer_accuracy(vector_search_idx))
print("리랭커 정답률    :", answer_accuracy(rerank_topk))

## Lab 11 · 미니 GraphRAG (관계를 뽑아 연결)
벡터 검색은 답이 여러 문서에 흩어진 질문("X-200 제조사의 대표는?")에 약합니다.
관계를 (주체, 관계, 대상) 트리플로 뽑아 그래프로 잇고, 경로를 따라 멀티홉으로 답합니다.

흐름: ① LLM 트리플 추출(JSON) → ② networkx 그래프 → ③ 질문 속 엔티티에서 2-hop 이웃 탐색 → ④ 경로를 근거로 답변

In [ ]:
import json
import networkx as nx

graph_docs = [
    "제품 X-200은 아크메전자가 제조한다.",
    "아크메전자의 대표이사는 김철수다.",
    "X-200의 배터리는 파워셀이 공급한다.",
    "김철수는 2024년 혁신경영 대상을 수상했다.",
]

def extract_triples(docs):
    prompt = ("각 문장에서 (주체, 관계, 대상) 트리플을 뽑아 JSON 배열로만 출력해. 설명 금지.\n"
              '예: [["X-200","제조사","아크메전자"]]\n\n' + "\n".join(docs))
    for attempt in range(2):                        # 파싱 실패 시 1회 재시도
        raw = ask(prompt, temperature=0).strip()
        if raw.startswith("```"):                   # ```json ... ``` 코드펜스 제거
            raw = raw.strip("`").strip()
            if raw[:4].lower() == "json":
                raw = raw[4:].strip()
        try:
            return [tuple(t) for t in json.loads(raw)]
        except json.JSONDecodeError:
            print(f"  JSON 파싱 실패(시도 {attempt+1}) → 재시도")
    raise RuntimeError("트리플 추출 실패 — 프롬프트의 'JSON 배열만' 지시를 확인하세요")

triples = extract_triples(graph_docs)
print("추출된 트리플:", triples)
G = nx.DiGraph()
for s, r, o in triples:
    G.add_edge(s, o, rel=r)
print("노드:", list(G.nodes))

### 그래프 시각화
작성한 지식그래프를 그림으로 확인합니다 — 노드(점)와 엣지(관계의 선).

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["AppleGothic", "Malgun Gothic", "NanumGothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(7, 5))
nx.draw(G, pos, with_labels=True, node_color="#DCE8FF", node_size=2600, font_size=11,
        edgecolors="#4A78D0", arrowsize=18)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "rel"), font_size=9)
plt.title("지식그래프"); plt.axis("off"); plt.show()

In [ ]:
def neighbors_context(G, start, hops=2):        # start에서 hops걸음 안의 관계(길)를 모두 수집
    seen, frontier, lines = {start}, {start}, []
    for _ in range(hops):
        nxt = set()
        for u in frontier:
            for _, v, d in G.out_edges(u, data=True):
                lines.append(f"{u} -[{d['rel']}]-> {v}"); nxt.add(v)
            for v, _, d in G.in_edges(u, data=True):
                lines.append(f"{v} -[{d['rel']}]-> {u}"); nxt.add(v)
        frontier = nxt - seen; seen |= nxt
    return "\n".join(dict.fromkeys(lines))

def graph_answer(query, hops=2):
    start = max((n for n in G.nodes if n in query), key=len, default=None)   # 질문 속 엔티티 찾기
    if start is None:
        return "질문에서 그래프 엔티티를 찾지 못함"
    ctx = neighbors_context(G, start, hops)
    print(f"[{start}에서 {hops}-hop 경로]\n{ctx}\n")
    return ask(f"[관계 정보]\n{ctx}\n\n[질문] {query}\n관계 정보만 근거로 짧게 답해줘.", temperature=0)

print("그래프 답:", graph_answer("X-200 제조사의 대표는 누구인가요?"))

> 도전 과제 — 슬라이드 '어떤 주제를 그래프로?'에서 고른 주제(조직도·부품 카탈로그·드라마 대본·뉴스 아카이브…)로
> `graph_docs`를 4~6문장 직접 써서 바꾸고, 멀티홉 질문을 하나 만들어 다시 실행해 보세요.

## 산출물 체크리스트
- [ ] 방대한 문서를 **파일에서 로드**해 청킹하고 메타데이터를 붙여 벡터DB에 저장했다
- [ ] 메타데이터 필터로 검색 범위를 좁혔다
- [ ] BM25 · 하이브리드(RRF) · Reranker 로 검색 품질을 끌어올렸다
- [ ] HyDE · Multi-Query 로 모호한 질문을 보강했다
- [ ] MMR 로 중복을 줄이고, 재배치로 강한 근거를 양 끝에 두었다
- [ ] Recall@k · MRR 로 검색을, LLM 심판으로 정답률을 숫자로 비교했다
- [ ] 미니 GraphRAG 로 흩어진 사실을 시각화하고 멀티홉으로 연결했다